In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs", "Property_ID"])
y = df["Price_INR_Lakhs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)],
    sparse_threshold=0
)

### Early stopping on vs off

In notebook 26 the 100 and 200 iteration runs gave the same score because early stopping
stopped both at 56 trees. Here we compare both modes for each setting.

In [2]:
def evaluate(params):
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", HistGradientBoostingRegressor(random_state=42, **params))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    return {
        **params,
        "MAE": mean_absolute_error(y_val, y_pred),
        "RMSE": mean_squared_error(y_val, y_pred) ** 0.5,
        "R2": r2_score(y_val, y_pred),
        "trees_used": model.named_steps["regressor"].n_iter_
    }

In [3]:
settings = [
    {"learning_rate": 0.10, "max_iter": 100},
    {"learning_rate": 0.10, "max_iter": 300},
    {"learning_rate": 0.05, "max_iter": 300},
    {"learning_rate": 0.05, "max_iter": 500},
]

results = []

for setting in settings:
    for early_stopping in [True, False]:
        results.append(evaluate({**setting, "early_stopping": early_stopping}))

early_stop_df = pd.DataFrame(results).sort_values("R2", ascending=False)
early_stop_df

,learning_rate,max_iter,early_stopping,MAE,RMSE,R2,trees_used
0,0.10,100,True,32.846547,68.672493,0.786996,56
2,0.10,300,True,32.846547,68.672493,0.786996,56
5,0.05,300,False,32.671548,68.708819,0.786771,300
4,0.05,300,True,32.834453,68.763963,0.786428,112
6,0.05,500,True,32.834453,68.763963,0.786428,112
1,0.10,100,False,32.690691,68.773679,0.786368,100
7,0.05,500,False,32.799066,68.960086,0.785208,500
3,0.10,300,False,33.060797,69.393202,0.782502,300


Without early stopping, more trees start to overfit (lr 0.10 with 300 trees is the worst).
With early stopping the model picks its own number of trees, so we keep it on and use
`learning_rate=0.05, max_iter=500` as the base for the next experiments.

### Tree size (max_leaf_nodes)

In [4]:
leaf_results = []

for max_leaf_nodes in [15, 31, 63, 127]:
    leaf_results.append(evaluate({
        "learning_rate": 0.05,
        "max_iter": 500,
        "max_leaf_nodes": max_leaf_nodes
    }))

leaf_df = pd.DataFrame(leaf_results).sort_values("R2", ascending=False)
leaf_df

,learning_rate,max_iter,max_leaf_nodes,MAE,RMSE,R2,trees_used
0,0.05,500,15,32.664654,68.333442,0.789094,150
1,0.05,500,31,32.834453,68.763963,0.786428,112
2,0.05,500,63,32.905429,68.853230,0.785874,94
3,0.05,500,127,32.977057,69.329412,0.782902,100


Smaller trees generalise better here: 15 leaves beats the default 31, and bigger trees
keep getting worse. We use `max_leaf_nodes=15` going forward.

### Regularisation (min_samples_leaf and l2_regularization)

In [5]:
reg_results = []

for min_samples_leaf in [20, 50, 100, 200]:
    for l2_regularization in [0.0, 1.0, 5.0]:
        reg_results.append(evaluate({
            "learning_rate": 0.05,
            "max_iter": 500,
            "max_leaf_nodes": 15,
            "min_samples_leaf": min_samples_leaf,
            "l2_regularization": l2_regularization
        }))

reg_df = pd.DataFrame(reg_results).sort_values("R2", ascending=False)
reg_df.head(8)

,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,MAE,RMSE,R2,trees_used
7,0.05,500,15,100,1.0,32.584237,68.062936,0.790761,162
9,0.05,500,15,200,0.0,32.526743,68.086866,0.790614,194
10,0.05,500,15,200,1.0,32.567853,68.101072,0.790526,177
3,0.05,500,15,50,0.0,32.654447,68.197597,0.789932,141
6,0.05,500,15,100,0.0,32.621852,68.209523,0.789859,150
8,0.05,500,15,100,5.0,32.645453,68.226498,0.789754,142
11,0.05,500,15,200,5.0,32.614807,68.233270,0.789712,143
1,0.05,500,15,20,1.0,32.642579,68.250711,0.789605,144


In [6]:
best = reg_df.iloc[0]

best_params = {
    "learning_rate": best["learning_rate"],
    "max_iter": int(best["max_iter"]),
    "max_leaf_nodes": int(best["max_leaf_nodes"]),
    "min_samples_leaf": int(best["min_samples_leaf"]),
    "l2_regularization": best["l2_regularization"]
}

print("Best parameters:", best_params)
print(f"Validation R2: {best['R2']:.4f}  (notebook 25 baseline: 0.7870)")

Best parameters: {'learning_rate': np.float64(0.05), 'max_iter': 500, 'max_leaf_nodes': 15, 'min_samples_leaf': 100, 'l2_regularization': np.float64(1.0)}
Validation R2: 0.7908  (notebook 25 baseline: 0.7870)


Tuned model: R2 **0.7908** vs **0.7870** for the notebook 25 baseline. The gain is small, so next we check with cross-validation that it is not just luck on this one split.